# ASS-ADE LoRA Training — Google Colab (Free T4)

This notebook fine-tunes a PEFT LoRA adapter on top of **TinyLlama-1.1B-Chat-v1.0**
using your locally-collected `training_data.jsonl`.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run `ass-ade train --collect` locally to produce `training_data/training_data.jsonl`
3. Upload the file when prompted in Cell 3

**After training (~15 min):**
- Download `lora_adapter.zip`
- Unzip into `models/lora_adapter/` in your project
- Run `ass-ade train --serve`

In [ ]:
# Cell 1 — Check GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU — training will be slow. Set Runtime > T4 GPU.')

In [ ]:
# Cell 2 — Install dependencies
!pip install -q transformers peft datasets accelerate safetensors huggingface_hub sentencepiece

In [ ]:
# Cell 3 — Upload training_data.jsonl
# Run `ass-ade train --collect` locally first, then upload the file here.
from google.colab import files
import os

print('Upload your training_data/training_data.jsonl file:')
uploaded = files.upload()
# Move to expected path
os.makedirs('training_data', exist_ok=True)
for name in uploaded:
    dest = f'training_data/{name}'
    with open(dest, 'wb') as f:
        f.write(uploaded[name])
    print(f'Saved to {dest}')

DATA_PATH = 'training_data/training_data.jsonl'
import json
n = sum(1 for line in open(DATA_PATH) if line.strip())
print(f'Training samples: {n}')

In [ ]:
# Cell 4 — Configuration
BASE_MODEL    = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
OUTPUT_DIR    = 'models/lora_adapter'
LORA_RANK     = 8
LORA_ALPHA    = 16
LORA_DROPOUT  = 0.05
EPOCHS        = 3
BATCH_SIZE    = 4
LEARNING_RATE = 2e-4
MAX_LENGTH    = 512

print(f'Base model : {BASE_MODEL}')
print(f'LoRA rank  : {LORA_RANK}')
print(f'Epochs     : {EPOCHS}')
print(f'Output     : {OUTPUT_DIR}')

In [ ]:
# Cell 5 — Load and prepare dataset
import json
from datasets import Dataset

def format_sample(s):
    instruction = s.get('instruction', '')
    input_ = s.get('input', '')
    output = s.get('output', '')
    prompt = f'### Instruction:\n{instruction}'
    if input_:
        prompt += f'\n\n### Input:\n{input_}'
    prompt += f'\n\n### Response:\n{output}'
    return prompt

samples = [json.loads(l) for l in open(DATA_PATH) if l.strip()]
texts = [format_sample(s) for s in samples]
ds = Dataset.from_dict({'text': texts})
print(f'Dataset: {len(ds)} samples')
print('\nExample (truncated):')
print(texts[0][:300])

In [ ]:
# Cell 6 — Load model and apply LoRA
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
# Cell 7 — Tokenize
def tokenize(batch):
    enc = tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH, padding=False)
    enc['labels'] = [ids.copy() for ids in enc['input_ids']]
    return enc

ds_tok = ds.map(tokenize, batched=True, remove_columns=['text'])
print(f'Tokenized: {len(ds_tok)} samples')

In [ ]:
# Cell 8 — Train
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR + '/checkpoints',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=2,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    logging_steps=10,
    save_steps=100,
    save_total_limit=1,
    fp16=True,
    gradient_checkpointing=True,
    report_to='none',
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_tok,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print(f'Training {len(ds_tok)} samples for {EPOCHS} epochs…')
trainer.train()

In [ ]:
# Cell 9 — Save adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Adapter saved to {OUTPUT_DIR}/')

import os
files_saved = list(os.listdir(OUTPUT_DIR))
print('Files:', files_saved)

In [ ]:
# Cell 10 — Download adapter as zip
import shutil
from google.colab import files

zip_path = 'lora_adapter'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print('Downloading lora_adapter.zip…')
files.download('lora_adapter.zip')
print('Done! Unzip to models/lora_adapter/ in your project, then run: ass-ade train --serve')

## Optional: Upload to HuggingFace Hub

To share the adapter or re-use it later, push it to your HF account:

```python
from huggingface_hub import HfApi
import os

# Set your HF token: Runtime > Secrets > HF_TOKEN
HF_TOKEN = os.environ.get('HF_TOKEN') or input('HF token: ')
REPO_ID = 'your-username/ass-ade-lora-v0'

api = HfApi()
api.create_repo(REPO_ID, exist_ok=True, token=HF_TOKEN)
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=REPO_ID,
    token=HF_TOKEN,
)
print(f'Adapter at: https://huggingface.co/{REPO_ID}')
```

Then load it locally:
```bash
ass-ade train --serve --adapter-dir models/lora_adapter
```